In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.stats.multitest as sm 
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


In [9]:
# Load and tidy data

df = pd.read_csv('../../raw_data/Glycosphingolipid_analysis/Deep-surface_GSLs.csv')
df['patient_id'] = list(df['id'].str.split('_', expand = True)[1].str.split('Deep', expand = True)[0].str.split('Surface', expand = True)[0])
df['type'] = np.where(df['id'].str.contains('Deep', case=False, na=False), 'Deep', 'Surface')
categories = ['2.16', 'GA2', '2.67', 'Gb3', 'GM3', '3.23', 'GM2','3.57', '3.61', 'GA1', '4.03', 'GM1a', 'GD3', 'GM1b', 'GD2', 'GD1a','5.24', '5.33', '5.45', 'GD1b', '5.99', 'GT1b', '6.84', '6.99', 'GQ1b','8.07']
df['z-score'] = df[categories].sum(axis = 1)
deep = df[df['type']=='Deep']
deep.columns = [col + '_deep' for col in deep.columns]
deep.rename(columns = {'id_deep':'id','group_deep':'group','patient_id_deep':'patient_id','type_deep':'type'}, inplace = True)
surface = df[df['type']=='Surface']
surface.columns = [col + '_surface' for col in surface.columns]
surface.drop(columns = ['group_surface', 'id_surface', 'type_surface', 'id_surface'], inplace = True)
surface.rename(columns = {'patient_id_surface':'patient_id'}, inplace = True)
surface

# Merge surface and deep data to facilitate pairwise comparison
paired = deep.merge(surface, left_on = 'patient_id', right_on = 'patient_id', how = 'inner')
PD_paired = paired[paired['group']=='PD']
Ctrl_paired = paired[paired['group']=='Control']
paired

/tmp/ipykernel_2316592/1476345141.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  deep.rename(columns = {'id_deep':'id','group_deep':'group','patient_id_deep':'patient_id','type_deep':'type'}, inplace = True)
/tmp/ipykernel_2316592/1476345141.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  surface.drop(columns = ['group_surface', 'id_surface', 'type_surface', 'id_surface'], inplace = True)
/tmp/ipykernel_2316592/1476345141.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_gu

,id,group,2.16_deep,GA2_deep,2.67_deep,Gb3_deep,GM3_deep,3.23_deep,GM2_deep,3.57_deep,...,5.33_surface,5.45_surface,GD1b_surface,5.99_surface,GT1b_surface,6.84_surface,6.99_surface,GQ1b_surface,8.07_surface,z-score_surface
0,1_P14/20Deep,Control,0.000201,0.009126,NaN,0.002207,0.025647,0.000467,0.030267,0.001530,...,0.005787,0.004466,0.470396,0.018230,0.268218,0.002209,0.002750,0.017034,0.000667,1.862103
1,1_P23/21Deep,Control,NaN,0.008130,NaN,NaN,0.020995,NaN,0.058361,NaN,...,0.004885,0.003053,0.337870,0.013257,0.185949,0.001257,0.001399,0.009268,NaN,1.427082
2,1_P48/15Deep,Control,0.000243,0.001372,0.000648,0.002009,0.025151,NaN,0.034039,NaN,...,0.006268,0.003622,0.541131,0.019742,0.299415,0.001590,NaN,0.016234,NaN,2.161986
3,1_P57/18Deep,Control,0.000579,0.002921,0.000864,0.002586,0.026219,NaN,0.032225,NaN,...,NaN,0.003682,0.373070,0.017058,0.222300,0.001217,0.001750,0.011182,NaN,1.809511
4,1_P58/10Deep,Control,NaN,0.017903,NaN,NaN,0.016756,NaN,0.043032,NaN,...,NaN,0.003769,0.443720,0.017469,0.260303,0.001442,0.001480,0.012349,NaN,1.994767
5,2_P13/21Deep,PD,0.000242,0.000013,0.000285,0.002157,0.024764,0.000319,0.019723,NaN,...,NaN,0.001892,0.196996,0.009847,0.131783,0.000554,0.000402,0.006072,NaN,1.075337
6,2_P16/16Deep,PD,0.000145,0.000196,0.000280,0.001839,0.026715,0.000370,0.033038,NaN,...,0.003744,0.002263,0.265460,0.010537,0.140739,0.000761,0.000405,0.020181,NaN,1.087527
7,2_P19/20Deep,PD,0.000454,0.022506,NaN,0.003081,0.029177,NaN,0.064629,NaN,...,NaN,0.003841,0.446273,0.019441,0.294677,0.001779,0.001391,0.017323,NaN,1.918334
8,2_P26/18Deep,PD,0.000214,0.002920,0.000317,0.002039,0.036798,0.000468,0.053034,0.001127,...,NaN,0.003786,0.374159,0.016586,0.229599,0.001045,0.001244,0.011178,NaN,2.079335
9,2_P65/19Deep,PD,NaN,NaN,NaN,NaN,0.019721,NaN,0.028175,NaN,...,NaN,0.004555,0.485066,0.019890,0.291512,0.001681,0.002463,0.016446,0.000210,2.051092


In [12]:
list_tests = []
for comp in ['z-score']:
    paired_filtered = PD_paired[[col for col in PD_paired.columns if comp in col]]
    paired_filtered.dropna(inplace = True)
    #print(comp)
    #print(np.shape(paired_filtered))
    if np.shape(paired_filtered)[0]<3:
        #print(f"Skipping {comp}: only {paired_filtered.shape[0]} samples")
        continue
    else:
        res = stats.ttest_rel(paired_filtered[paired_filtered.columns[0]], paired_filtered[paired_filtered.columns[1]])
        #res = wilcoxon(paired_filtered[paired_filtered.columns[0]], paired_filtered[paired_filtered.columns[1]])
        list_tests.append(comp)
        total_res = pd.DataFrame(res).T



total_res.rename(columns = {0:'statistic',1:'pval'}, inplace = True)
total_res.index = list_tests
total_res.dropna(inplace = True)
total_res['padj'] = sm.fdrcorrection(list(total_res['pval']), alpha=0.05, method='indep', is_sorted=False)[1]
#total_res.to_csv('../../processed_data/09_validation_lipids/paired_GSL_analysis.csv')
sig = total_res[total_res['pval']<0.05]

/tmp/ipykernel_2316592/3432284344.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  paired_filtered.dropna(inplace = True)


In [13]:
total_res['padj']

z-score    0.219751
Name: padj, dtype: float64